<a href="https://colab.research.google.com/github/pramodkumarw/Github-Colab/blob/main/checkpointer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langgraph

In [ ]:
# STEP 1: SET UP THE ENVIRONMENT

# STEP 1.1 INSTALL THE REQUIRED PACKAGES
!pip install langchain_community

!pip install langchain-groq
!pip install langchain
!pip install langgrah

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
# from rich import print
from google.colab import userdata

In [3]:

GROQ_API_KEY=userdata.get("GROQ_API_KEY")
llm=ChatGroq(model="openai/gpt-oss-120b", groq_api_key=GROQ_API_KEY)


In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str


In [5]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    return {'joke': response}

def generate_explanation(state: JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the workplace had great “slice” benefits and it wanted a “topping” career! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it literally means | The wordplay / pun |\n|------------------|------------------------|--------------------|\n| **“Why did the pizza apply for a job?”** | Sets up a classic “why‑did‑the‑X…?” joke format. The “X” here is a pizza, which of course can’t look for work. | The absurdity of a food item behaving like a person creates the comedic premise. |\n| **“Because it heard the workplace had great ‘slice’ benefits…”** | In a normal job, “benefits” are perks like health insurance, vacation, etc. | “Slice” is a pun on “share” or “piece.” In a pizza context a slice is a portion of the pizza, so “slice benefits” sounds like “nice benefits” while also suggesting the pizza would get a literal slice of something (like a piece of the profit). |\n| **“…and it w

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the workplace had great “slice” benefits and it wanted a “topping” career! 🍕😄', 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it literally means | The wordplay / pun |\n|------------------|------------------------|--------------------|\n| **“Why did the pizza apply for a job?”** | Sets up a classic “why‑did‑the‑X…?” joke format. The “X” here is a pizza, which of course can’t look for work. | The absurdity of a food item behaving like a person creates the comedic premise. |\n| **“Because it heard the workplace had great ‘slice’ benefits…”** | In a normal job, “benefits” are perks like health insurance, vacation, etc. | “Slice” is a pun on “share” or “piece.” In a pizza context a slice is a portion of the pizza, so “slice benefits” sounds like “nice benefits” while also suggesting the pizza would get a literal slice of something (like a piece of the profit)

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the workplace had great “slice” benefits and it wanted a “topping” career! 🍕😄', 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it literally means | The wordplay / pun |\n|------------------|------------------------|--------------------|\n| **“Why did the pizza apply for a job?”** | Sets up a classic “why‑did‑the‑X…?” joke format. The “X” here is a pizza, which of course can’t look for work. | The absurdity of a food item behaving like a person creates the comedic premise. |\n| **“Because it heard the workplace had great ‘slice’ benefits…”** | In a normal job, “benefits” are perks like health insurance, vacation, etc. | “Slice” is a pun on “share” or “piece.” In a pizza context a slice is a portion of the pizza, so “slice benefits” sounds like “nice benefits” while also suggesting the pizza would get a literal slice of something (like a piece of the profit